# 15 · Scaling sweeps in D, T and d_max  (~6–10 h total; each sweep runs on its own)

> Part of the October 2026 experiments (notebooks 10–19), run with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, full repetitions even for slow cells, explicit thread counts, failures recorded. Never run two of these notebooks at the same time.

Feeds §6.4 (new figure). Quest-model data from `exp_data.quest` (vectorised version of
`bench/agrawal_generator.py`: same model, different random stream), s = 1e-4, k ∈ {2,3,4}.

* **D sweep**: T=20, I=4, N=50K, L=50K, corr 0.5; D ∈ {0.1, 0.25, 0.5, 1, 2.5, 5} M.
  P2 predicts the 1-thread sparse layout loses ground to the triangular matrix as D grows (ρ ∝ 1/D).
* **T sweep**: D=500K, N=25K, I=4, L=25K; T ∈ {5, 10, 20, 30, 50}.
* **d_max sweep**: base T=10, D=500K, N=25K; `inject_tail` raises d_max ∈ {50, 200, 500,
  1000, 2500} at fixed D and d_avg (d_CV rises with it, as it must). Tests the C(d_max, k−1) term.

Set `RUN_SWEEPS`. Generation is cached in `data/generated/`; delete a parquet to regenerate.

Outputs: `results/runs/<campaign>/scaling_D.csv`, `scaling_T.csv`, `scaling_dmax.csv`.

`FA_MODE=replay` analyses the saved campaign (`vm_20261029` unless `FA_CAMPAIGN` says otherwise): no mining and no data needed; tables and figures go to `results/derived/<campaign>/`.


In [ ]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
REPLAY = os.environ.get("FA_MODE") == "replay"    # analyse the saved campaign: no mining, nothing overwritten
os.environ.setdefault("FA_CAMPAIGN", "vm_20261029" if REPLAY else "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
OUT = BUNDLE / "results" / "derived" / pr.CAMPAIGN if REPLAY else pr.RESULTS
OUT.mkdir(parents=True, exist_ok=True)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

In [ ]:
RUN_SWEEPS = [] if REPLAY else ["dmax", "D", "T"]     # d_max first: it is the decisive one
METHODS = ["hyb", "hyb_1t", "hyb_sfx_1t", "hyb_seeded", "hyb_seeded_1t",
           "pyfim_eclat", "pyfim_fpgrowth", "borgelt_apriori", "borgelt_eclat", "lcm"]
KS, S = [2, 3, 4], 1e-4

def make(name, gen):
    if not pdd.is_registered(name):
        t0 = time.time(); df = gen()
        st = pdd.cost_stats(df, (S,))[0]
        pdd.register_parquet(name, df, meta=st)
        print(f"generated {name}: {len(df):,} rows in {time.time()-t0:.0f}s, d_avg={st['d_avg']:.2f}, d_max={st['d_max']}")
    return name

In [ ]:
if "dmax" in RUN_SWEEPS:
    base = pdd.quest(D=500_000, T=10, I=4, L=25_000, N=25_000, seed=11)
    names = [make(f"sweep_dmax_{dm}", lambda dm=dm: pdd.inject_tail(base, d_max=dm, n_wide=100))
             for dm in (50, 200, 500, 1000, 2500)]
    pr.run_grid("scaling_dmax", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

In [ ]:
if "D" in RUN_SWEEPS:
    names = [make(f"sweep_D_{D}", lambda D=D: pdd.quest(D=D, T=20, I=4, L=50_000, N=50_000, seed=12))
             for D in (100_000, 250_000, 500_000, 1_000_000, 2_500_000, 5_000_000)]
    pr.run_grid("scaling_D", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

In [ ]:
if "T" in RUN_SWEEPS:
    names = [make(f"sweep_T_{T}", lambda T=T: pdd.quest(D=500_000, T=T, I=4, L=25_000, N=25_000, seed=13))
             for T in (5, 10, 20, 30, 50)]
    pr.run_grid("scaling_T", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

In [ ]:
for exp in ("scaling_dmax", "scaling_D", "scaling_T"):
    s = pr.summarize(pr.load_results(exp))
    if len(s):
        display(exp, s.pivot_table(index=["dataset", "k"], columns="method", values="wall_s").round(2))